In [ ]:
!pip install tf-agents[reverb]
!export TF_USE_LEGACY_KERAS=1
!pip install keras==2.15 --user

In [ ]:
!pip install tf-agents==0.3.0
!pip install --user dm-reverb
!pip install tensorflow==2.11.0
!pip install keras==2.15.0 --user

In [ ]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import abc
import tensorflow as tf

from tf_agents.environments import py_environment
from tf_agents.environments import tf_environment
from tf_agents.environments import tf_py_environment
from tf_agents.environments import utils
from tf_agents.specs import array_spec
from tf_agents.environments import wrappers
from tf_agents.trajectories import time_step as ts
from tf_agents.networks import sequential
import reverb
from tf_agents.agents.dqn import dqn_agent
from tf_agents.utils import common
from tf_agents.specs import tensor_spec
from tf_agents.replay_buffers import reverb_replay_buffer
from tf_agents.replay_buffers import reverb_utils
from tf_agents.policies import py_tf_eager_policy
from tf_agents.drivers import py_driver
#
import base64
import imageio
import IPython
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import PIL.Image
import reverb

import tensorflow as tf

from tf_agents.agents.dqn import dqn_agent
from tf_agents.drivers import py_driver
from tf_agents.environments import suite_gym
from tf_agents.environments import tf_py_environment
from tf_agents.eval import metric_utils
from tf_agents.metrics import tf_metrics
from tf_agents.networks import sequential
from tf_agents.policies import py_tf_eager_policy
from tf_agents.policies import random_tf_policy
from tf_agents.replay_buffers import reverb_replay_buffer
from tf_agents.replay_buffers import reverb_utils
from tf_agents.trajectories import trajectory
from tf_agents.specs import tensor_spec
from tf_agents.utils import common

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
from IPython.display import HTML
from base64 import b64encode
import itertools

In [ ]:
#get inertia tensor for drone from drive (calculated with Inertiacalc.ipynb)
inertia = np.load('drive/MyDrive/RAINinertia.npy')
print(inertia)

In [ ]:
#represent quaternions as (float,np.array)
#define multiplication
def quatmul(q1,q2):
    s1,v1 = q1
    s2,v2 = q2
    ns = s1*s2-np.dot(v1,v2)
    nv = s1*v2 +s2*v1 + np.cross(v1,v2)
    return (ns,nv)

#return inverse of quaternion for rotation formula:
def quatinv(q):
    s,v = q
    return (s,-v)

#using quaternions to rotate a position vector:
def quatrot(q,p):
    pquat = (0,p)
    return quatmul(quatmul(q,pquat),quatinv(q))[1]

#normalise quaternion to prevent numerical drift:
def quatnorm(q):
    s,v = q
    correction = 1/np.sqrt(np.dot(v,v) + s*s)
    return (s*correction,v*correction)

#convert quaternion to rotational matrix (for getting inertia tensor at current time):
def quatconv(q):
    s,v = q
    vx,vy,vz = tuple(v)
    return np.matrix([[1-2*(vy**2)-2*(vz**2), 2*vx*vy-2*s*vz, 2*vx*vz+2*s*vy],
                      [2*vx*vy+2*s*vz, 1-2*(vx**2)-2*(vz**2), 2*vy*vz-2*s*vx],
                      [2*vx*vz-2*s*vy, 2*vy*vz+2*s*vx, 1-2*(vx**2)-2*(vy**2)]])

#for debugging
#return quaternion corresponding to axis angle rotation (with angle in radians, vector is np.array):
def quatfromAA(axvec,theta):
    #normalise the vector:
    correction = 1/np.sqrt(np.dot(axvec,axvec))
    axvec = axvec * correction
    theta = theta/2
    return (np.cos(theta),np.sin(theta)*axvec)

In [ ]:
#z axis is upwards!
class CuboidEnv(py_environment.PyEnvironment):
    def __init__(self,propstates,reclist = []):
        #CHANGE LATER with continuous action space model
        self._propstates = propstates
        #make folder to store simulation renders
        self._foldername = '/simrender'
        if not os.path.exists(self._foldername):
            os.makedirs(self._foldername)
        os.chdir(self._foldername)
        for f in os.listdir('/simrender'):
            os.remove(f)
        #only store data for some select episodes:
        self._store_eps = reclist
        #bounds for randomised goal points:
        self.goal_upperbound = 5
        self.goal_lowerbound = -5
        #define maximum distance drone can move from goal before ending the episode
        self._max_dist = 10
        #if true reset to initial parameters - triggered when episode is too long or model has reached goal point:
        self._episode_ended = False
        #end episode if step count goes beyond max step count:
        self._step_count = 0
        self._max_step_count = 400
        self._ep_count = 0
        self._cum_reward = 0
        #length of each episode for retrieving renders
        self._ep_lens = []
        #define timestep for simulation:
        self._sim_timestep = 0.0125
        #define multiple of this as training step for model:
        self._train_timestep = 0.025
        #constant environment properties:
        #cuboid dimensions
        self.a = 0.1
        self.b = 0.1
        self.c = 0.05
        self.mass = 0.05

        #modelling drone as cuboid gives: CHANGE LATER
        self.inertia = np.diagflat(np.matrix([1/12*(self.b**2+self.c**2),1/12*(self.a**2+self.c**2),1/12*(self.a**2+self.b**2)])) * self.mass
        self.inv_inertia = np.linalg.inv(self.inertia)

        #cap on vel CHANGE LATER:
        self.vel_cap = 30
        self.rot_mot_cap = 3
        #assuming diagonalised inertia tensor:
        self.angular_mom_cap = max(self.inertia[0,0],self.inertia[1,1],self.inertia[2,2]) * self.rot_mot_cap

        #define output from this environment - 3D goal vector pointing to desired point, 4D quaternion orientation vector, 3D velocity vector, 3D angular momentum vector
        self._observation_spec = array_spec.BoundedArraySpec(shape=(13,), dtype=np.float32,
        minimum=[-1 for _ in range(13)],
        maximum=[1 for _ in range(13)], name='observation')

        #define input to this environment - 4 scalars corresponding the the force of the 4 propellers
        self._action_spec = array_spec.BoundedArraySpec(shape=(), dtype=np.int32, minimum = 0, maximum = len(self._propstates)-1,  name='action')
        #starting environment state:
        self.pos = np.array([0.,0.,0.])
        self.vel = np.array([0.,0.,0.])
        self.qorient = (1,np.array([0.,0.,0.]))
        self.rot_mot = np.array([0.,0.,0.])
        morient = quatconv(self.qorient)
        self.angular_mom = np.squeeze(np.asarray(morient @ self.inertia @ morient.transpose() @ self.rot_mot))

        #positions of the propellers
        self.props  = [np.array([self.a/2,self.b/2,self.c/2]),np.array([-self.a/2,self.b/2,self.c/2]),np.array([-self.a/2,-self.b/2,self.c/2]),np.array([self.a/2,-self.b/2,self.c/2])]
        #define point for the drone to reach
        self._goalpoint = np.random.uniform(self.goal_lowerbound,self.goal_upperbound,3)
        #get the vector between current and goal point
        self._goalvec = self._goalpoint - self.pos
        #define a tolerance which determines how close the drone should be to goal point to end the episode
        self._goaltol = 1.4
        #keep track of the distance to goal point for the reward function
        self._goaldist = np.sqrt(np.dot(self._goalvec,self._goalvec))

    def action_spec(self):
        return self._action_spec

    def observation_spec(self):
        return self._observation_spec

    def _reset(self):
        self.pos = np.array([0.,0.,0.])
        self.vel = np.array([0.,0.,0.])
        self.qorient = (1,np.array([0.,0.,0.]))
        self.rot_mot = np.array([0.,0.,0.])
        morient = quatconv(self.qorient)
        self.angular_mom = np.squeeze(np.asarray(morient @ self.inertia @ morient.transpose() @ self.rot_mot))
        self._episode_ended = False
        self._goalpoint = np.random.uniform(-5,5,3)
        self._goalvec = self._goalpoint - self.pos
        self._goaldist = np.sqrt(np.dot(self._goalvec,self._goalvec))
        self._step_count = 0
        self._cum_reward = 0
        self._ep_count += 1
        return ts.restart(self._statevec())

    #step providing model feedback
    def _step(self,propnum):
        propforces = np.array(self._propstates[propnum])
        self._propforces = propforces
        if self._episode_ended:
            return self.reset()
        iters = self._train_timestep/self._sim_timestep
        #iterate through simulation steps
        for i in range(int(iters)):
            self._microstep(self._sim_timestep,propforces)
        self._step_count += 1
        #calculate reward based on progress to goal:
        prevgoaldist = self._goaldist
        self._goalvec = self._goalpoint - self.pos
        self._goaldist = np.sqrt(np.dot(self._goalvec,self._goalvec))
        reward = prevgoaldist - self._goaldist
        #stop episode if max step count has been exceeded or drone has reached goal point:
        if self._goaldist<=self._goaltol:
            self._episode_ended = True
            reward += 50
            #print('achieved goal')
        elif self._goaldist>self._max_dist:
            #print(self._goaldist,self._max_dist)
            self._episode_ended = True
            reward -= 50
            #print('went too far from goal')
            #ensure final state return isnt outside bounds
            self._goalvec = np.clip(self._goalvec,-self._max_dist,self._max_dist)
        elif self._step_count>self._max_step_count:
            self._episode_ended = True
            reward -= 50
            #print('stepcount exceeded')
        self._cum_reward += reward
        if self._ep_count in self._store_eps:
            self._write3D(reward)
        if self._episode_ended:
            self._ep_lens.append(self._step_count)
            if self._store_eps != []:
                if self._ep_count == max(self._store_eps):
                    self._render()
            return ts.termination(self._statevec(), reward)
        else:
            return ts.transition(self._statevec(), reward = reward, discount = 0.99)

    #current state of environment
    def _statevec(self):
        s,v = self.qorient
        #for better performance return all values normalised between -1 and 1
        normgoal = self._goalvec/self._max_dist
        normvel = self.vel/self.vel_cap
        normmom = self.angular_mom/self.angular_mom_cap
        state = np.float32(np.concatenate((normgoal,np.array([s]),v,normvel,normmom)))
        return state

    #step for the simulation
    def _microstep(self,h,propforces):
        #propforces = [np.array([0,0,5]),np.array([0,0,5]),np.array([0,0,2]),np.array([0,0,2])]
        #linear motion
        self.pos += h*self.vel

        #get forces in terms of world coordinate system
        worldpropforces = [quatrot(self.qorient,np.array([0,0,propforce])) for propforce in propforces]

        #get resultant force and thus acceleration, accounting for gravity
        resforce = sum(worldpropforces) + np.array([0,0,-self.mass * 9.81])
        accel = resforce/self.mass

        self.vel += h*accel
        #cap velocity
        vel_mag = np.sqrt(np.dot(self.vel,self.vel))
        if vel_mag > self.vel_cap:
            self.vel = (self.vel_cap/vel_mag)*self.vel

        #rotational motion
        #get torque as cross product of location of force, force in body coordinates then convert to world coordinates
        torquesum = np.array([0.,0.,0.])
        for prop,propforce in zip(self.props,propforces):
            torquesum += quatrot(self.qorient,np.cross(prop,np.array([0,0,propforce])))

        #rotational motion
        morient = quatconv(self.qorient)
        self.rot_mot = np.squeeze(np.asarray(morient @ self.inv_inertia @ morient.transpose() @ self.angular_mom))
        #rot_mag = np.sqrt(np.dot(self.rot_mot,self.rot_mot))
        #if rot_mag > self.rot_mot_cap:
            #self.rot_mot = (self.rot_mot_cap/rot_mag)*self.rot_mot
        #update orientation quaternion
        ds,dv = quatmul((0,self.rot_mot),self.qorient)
        s,v = self.qorient
        self.qorient = (s+ds*0.5*h,v+dv*0.5*h)
        #normalise to prevent numerical drift
        self.qorient = quatnorm(self.qorient)

        #torque is the derivative of angular momentum
        self.angular_mom += h*torquesum
        #hard cap angular momentum CHANGE LATER
        mom_mag = np.sqrt(np.dot(self.angular_mom,self.angular_mom))
        if mom_mag > self.angular_mom_cap:
            self.angular_mom = (self.angular_mom_cap/mom_mag)*self.angular_mom

    #write render to file
    def _write3D(self,reward):
        fig = plt.figure(figsize=(5,5))
        ax = plt.axes(projection='3d')#f'linvel:{self.vel}\n, angvel:{self.rot_mot}\n, angmom:{self.angular_mom}\n, props:{self._propforces}' transform=ax.transAxes
        plt.title(f'linvel:{self.vel}\nangvel:{self.rot_mot}\nangmom:{self.angular_mom}\nprops:{self._propforces}\nreward:{reward}\ngoaldist:{self._goaldist}',loc='left', y=0.9, fontsize=9)
        upper = self._max_dist + self.goal_upperbound
        lower = self.goal_lowerbound - self._max_dist
        ax.set_xlim([lower, upper])
        ax.set_ylim([lower, upper])
        ax.set_zlim([lower, upper])
        ax.scatter([self.pos[0],self._goalpoint[0]],[self.pos[1],self._goalpoint[1]],[self.pos[2],self._goalpoint[2]], c=['b','r'])
        fname1 = '1-'+str(self._ep_count)+'-'+str(self._step_count)+'.png'
        plt.savefig(fname1) #,bbox_inches='tight'
        ax.cla()
        plotpoints = np.array([quatrot(self.qorient,point).tolist() for point in self.props])
        plt.title(f'epnum:{self._ep_count}\ncum_reward:{self._cum_reward}',loc='left', y=0.9, fontsize=15)
        ax.set_xlim([-0.1, 0.1])
        ax.set_ylim([-0.1, 0.1])
        ax.set_zlim([-0.1, 0.1])
        ax.scatter(plotpoints[:, 0], plotpoints[:, 1], plotpoints[:, 2])
        fname2 = '2-'+str(self._ep_count)+'-'+str(self._step_count)+'.png'
        plt.savefig(fname2)
        plt.close()

    #put frames together into video, save avi and mp3 versions in /simrender, called on termination of last episode in store_eps
    def _render(self):
        firstframe = cv2.imread(f'{self._store_eps[0]}-1-1.png')
        h,w,_ = np.shape(firstframe)
        fps = int(1/self._train_timestep) #self._sim_timestep if updating every iteration
        vid = cv2.VideoWriter('trainrender.avi',cv2.VideoWriter_fourcc(*'MJPG'),fps,(w*2,h))
        for epnum in self._store_eps: #self._ep_lens
            eplen = self._ep_lens[epnum - 1]
            for stepnum in range(1,eplen+1):
                frame1 = cv2.imread('1-'+str(epnum)+'-'+str(stepnum)+'.png')
                frame2 = cv2.imread('2-'+str(epnum)+'-'+str(stepnum)+'.png')
                fullframe = np.concatenate((frame1,frame2),axis=1)
                vid.write(fullframe)
        cv2.destroyAllWindows()
        vid.release()
        os.system('ffmpeg -i trainrender.avi \-c:v libx264 \-pix_fmt yuv420p \-preset medium \-crf 23 \-movflags +faststart \-y trainnew.mp4')

In [ ]:
propstates = list(itertools.product([0.36,0.37], repeat = 4)) + [(0.2,0.2,0.2,0.2),(0.1,0.1,0.1,0.1),(0,0,0,0),(0.3,0.3,0.3,0.3)]
environment = CuboidEnv(propstates,reclist = [1,2])
utils.validate_py_environment(environment, episodes=2)

In [ ]:
#get rendered video
mp4 = open('trainnew.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=1000 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

In [ ]:
#hyperparameters
num_iterations = 250000
initial_collect_steps = 100
collect_steps_per_iteration = 1
replay_buffer_max_length = 100000
batch_size = 64
learning_rate = 1e-3
log_interval = 200
num_eval_episodes = 5
eval_interval = 10000

In [ ]:
#propstates = list(itertools.product([0.38,0.4], repeat = 4)) + [(0.2,0.2,0.2,0.2),(0.1,0.1,0.1,0.1),(0,0,0,0)]
propstates = list(itertools.product([0.41,0.45], repeat = 4)) + [(0.2,0.2,0.2,0.2),(0.1,0.1,0.1,0.1),(0,0,0,0),(0.3,0.3,0.3,0.3)]
train_py_env = CuboidEnv(propstates)
eval_py_env = CuboidEnv(propstates) #reclist = list(range(1,int((num_iterations/eval_interval)*num_eval_episodes),num_eval_episodes))
env = CuboidEnv(propstates)
train_env = tf_py_environment.TFPyEnvironment(train_py_env)
eval_env = tf_py_environment.TFPyEnvironment(eval_py_env)

In [ ]:
fc_layer_params = (200, 200)

def dense_layer(num_units):
  return tf.keras.layers.Dense(
      num_units,
      activation=tf.keras.activations.relu,
      kernel_initializer=tf.keras.initializers.VarianceScaling(
          scale=2.0, mode='fan_in', distribution='truncated_normal'))

dense_layers = [dense_layer(num_units) for num_units in fc_layer_params]
q_values_layer = tf.keras.layers.Dense(
    len(propstates),
    activation=None,
    kernel_initializer=tf.keras.initializers.RandomUniform(
        minval=-0.03, maxval=0.03),
    bias_initializer=tf.keras.initializers.Constant(-0.2))
q_net = sequential.Sequential(dense_layers + [q_values_layer])

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
train_step_counter = tf.Variable(0)
agent = dqn_agent.DqnAgent(
    train_env.time_step_spec(),
    train_env.action_spec(),
    q_network=q_net,
    optimizer=optimizer,
    td_errors_loss_fn=common.element_wise_squared_loss,
    train_step_counter=train_step_counter)
agent.initialize()

In [ ]:
eval_policy = agent.policy
collect_policy = agent.collect_policy

In [ ]:
def compute_avg_return(environment, policy, num_episodes=10):
    total_return = 0.0
    for _ in range(num_episodes):
        time_step = environment.reset()
        episode_return = 0.0
        while not time_step.is_last():
            action_step = policy.action(time_step)
            time_step = environment.step(action_step.action)
            episode_return += time_step.reward
            #print(time_step.reward)
        total_return += episode_return
    avg_return = total_return / num_episodes
    return avg_return.numpy()[0]

In [ ]:
table_name = 'uniform_table'
replay_buffer_signature = tensor_spec.from_spec(
      agent.collect_data_spec)
replay_buffer_signature = tensor_spec.add_outer_dim(
    replay_buffer_signature)

table = reverb.Table(
    table_name,
    max_size=replay_buffer_max_length,
    sampler=reverb.selectors.Uniform(),
    remover=reverb.selectors.Fifo(),
    rate_limiter=reverb.rate_limiters.MinSize(1),
    signature=replay_buffer_signature)

reverb_server = reverb.Server([table])

replay_buffer = reverb_replay_buffer.ReverbReplayBuffer(
    agent.collect_data_spec,
    table_name=table_name,
    sequence_length=2,
    local_server=reverb_server)

rb_observer = reverb_utils.ReverbAddTrajectoryObserver(
    replay_buffer.py_client,
    table_name,
    sequence_length=2)

In [ ]:
dataset = replay_buffer.as_dataset(
    num_parallel_calls=3,
    sample_batch_size=batch_size,
    num_steps=2).prefetch(3)

iterator = iter(dataset)

In [ ]:
#21 min for 80000 steps
agent.train = common.function(agent.train)
# Reset the train step
agent.train_step_counter.assign(0)

# Evaluate the agent's policy once before training
avg_return = compute_avg_return(eval_env, agent.policy, num_eval_episodes)
returns = [avg_return]

# Reset the environment
time_step = train_py_env.reset()

# Create a driver to collect experience
collect_driver = py_driver.PyDriver(
    train_py_env,
    py_tf_eager_policy.PyTFEagerPolicy(
      agent.collect_policy, use_tf_function=True),
    [rb_observer],
    max_steps=collect_steps_per_iteration)
initial_driver = py_driver.PyDriver(
    train_py_env,
    py_tf_eager_policy.PyTFEagerPolicy(
      agent.collect_policy, use_tf_function=True),
    [rb_observer],
    max_steps=initial_collect_steps)
time_step, _ = initial_driver.run(time_step)
for _ in range(num_iterations):
    # Collect a few steps and save to the replay buffer
    time_step, _ = collect_driver.run(time_step)

    # Sample a batch of data from the buffer and update the agent's network
    experience, unused_info = next(iterator)
    train_loss = agent.train(experience).loss
    step = agent.train_step_counter.numpy()
    if step % log_interval == 0:
        print('step = {0}: loss = {1}'.format(step, train_loss))

    if step % eval_interval == 0:
        avg_return = compute_avg_return(eval_env, agent.policy, num_eval_episodes)
        print('step = {0}: Average Return = {1}'.format(step, avg_return))
        returns.append(avg_return)

In [ ]:
#get rendered video
#os.system('ffmpeg -i trainrender.avi \-c:v libx264 \-pix_fmt yuv420p \-preset medium \-crf 23 \-movflags +faststart \-y trainnew.mp4')
mp4 = open('trainnew.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=1000 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

In [ ]:
newenv = tf_py_environment.TFPyEnvironment(CuboidEnv(propstates,reclist = [1,2,3]))
for _ in range(1,4):
    time_step = newenv.reset()
    while not time_step.is_last():
        action_step = agent.policy.action(time_step)
        time_step = newenv.step(action_step.action)


In [ ]:
firstframe = cv2.imread(f'1-1-1.png')
h,w,_ = np.shape(firstframe)
fps = int(1/0.05) #self._sim_timestep if updating every iteration
vid = cv2.VideoWriter('trainrender.avi',cv2.VideoWriter_fourcc(*'MJPG'),fps,(w*2,h))
for eplen,epnum in zip([395,396,378,390,471],[1,2,3,4,5]): #self._ep_lens
    for stepnum in range(1,eplen+1):
        frame1 = cv2.imread('1-'+str(epnum)+'-'+str(stepnum)+'.png')
        frame2 = cv2.imread('2-'+str(epnum)+'-'+str(stepnum)+'.png')
        print('2-'+str(epnum)+'-'+str(stepnum)+'.png')
        fullframe = np.concatenate((frame1,frame2),axis=1)
        vid.write(fullframe)
cv2.destroyAllWindows()
vid.release()